In [1]:
import sys
!{sys.executable} -m pip install PyMuPDF
!{sys.executable} -m pip install torch torchvision torchaudio

# 1. Data Loading & Preprocessing
Brief: This section loads NDA documents and preprocesses them into clauses.

In [2]:
import fitz  # PyMuPDF

def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# Load files
nda1_text = extract_text_from_pdf("NDA-1.pdf")
nda2_text = extract_text_from_pdf("NDA-2.pdf")

# Preview
print(nda1_text[:1000])

Classroom Use of Confidential Information 
Non-Disclosure Agreement (NDA) Template Package 
– for Course Instructors 
 
 
 
 
 
Do not use the attached template if students 
are receiving third party proprietary 
confidential information: 
 …as part of a student placement, internship, 
co-curricular or other off-site activity. 
Contact the Office of the Vice-Provost, 
Students for applicable templates and 
additional information related to these 
activities; or 
 …to conduct research under the supervision 
of a principal investigator. Contact the 
Innovations & Partnerships Office for 
applicable templates and additional 
information related to supervised research. 
 
Use the attached template if: 
□ You have determined that receiving 
confidential information from a third party 
for classroom use would be highly beneficial 
to your students enrolled in the academic 
course you are teaching; 
□ You have obtained approval from your 
academic unit/division head, for you and 
your stude

In [3]:
import re

def clean_text(text):
    # remove weird line breaks
    text = text.replace("\n", " ")
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    # remove weird symbols
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    
    return text.strip()

nda1_clean = clean_text(nda1_text)
nda2_clean = clean_text(nda2_text)

print(nda1_clean[:1000])

Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your students to receive confidential information from

In [4]:
def split_into_clauses(text):
    # split based on numbered sections (1. 2. 3. etc.)
    clauses = re.split(r'\s(?=\d+\.)', text)
    
    # remove tiny/noisy chunks
    clauses = [c.strip() for c in clauses if len(c.strip()) > 50]
    
    return clauses

nda1_clauses = split_into_clauses(nda1_clean)
nda2_clauses = split_into_clauses(nda2_clean)

print("Number of clauses:", len(nda1_clauses))
print("\nExample clause:\n", nda1_clauses[0])

Number of clauses: 19

Example clause:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your students 

In [5]:
import sys
!{sys.executable} -m pip uninstall -y transformers
!{sys.executable} -m pip install transformers[sentencepiece] --upgrade --force-reinstall

Found existing installation: transformers 5.5.4
Uninstalling transformers-5.5.4:
  Successfully uninstalled transformers-5.5.4
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3

In [6]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.11.0+cu130
GPU available: True


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_text(text):
    input_text = "summarize: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=20
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 9133.51it/s]


In [8]:
sample_clause = nda1_clauses[1][:500]

print("ORIGINAL:\n", sample_clause)
print("\nSUMMARY:\n", summarize_text(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

SUMMARY:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.


# 2. Feature 1: Summarization

## Baseline
Uses T5-small model for basic summarization.

## Improved
Uses prompt engineering and beam search to improve output quality.

## Comparison
Outputs from both approaches are compared.

In [9]:
def summarize_text_improved(text):
    input_text = "summarize key points: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=25,
        num_beams=4,           # better quality
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
print("BASELINE:\n", summarize_text(sample_clause))
print("\nIMPROVED:\n", summarize_text_improved(sample_clause))

BASELINE:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.

IMPROVED:
 the academic unit/division head with signing authority is identified. the academic unit/division head is not typically authorized to execute an NDA.


# 3. Feature 2: Risk Detection

## Baseline (Zero-shot)
Uses a pre-trained model to classify clauses as high or low risk.

## Improved (Logistic Regression)
Uses a trained machine learning model on labeled data.

## Comparison
Predictions from both models are compared.

In [11]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 8785.72it/s]


In [12]:
labels = ["high risk", "low risk"]

result = classifier(sample_clause, candidate_labels=labels)

print(result)

{'sequence': '1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.', 'labels': ['high risk', 'low risk'], 'scores': [0.8147278428077698, 0.18527214229106903]}


In [13]:
data = [
    ("Confidential information must not be disclosed to third parties", "high risk"),
    ("Recipient must protect confidential data", "high risk"),
    ("All confidential information must be returned after termination", "high risk"),
    ("This agreement lasts for two years", "low risk"),
    ("The agreement is governed by local law", "low risk"),
    ("Students may use the information only for coursework", "low risk"),
]

texts = [x[0] for x in data]
labels = [x[1] for x in data]

In [14]:
import sys
!{sys.executable} -m pip install scikit-learn

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [16]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X, labels)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [17]:
test_X = vectorizer.transform([sample_clause])
prediction = model_lr.predict(test_X)

print("Prediction:", prediction[0])

Prediction: low risk


In [18]:
for clause in nda1_clauses[:5]:
    print("\nCLAUSE:\n", clause[:200])
    
    # Zero-shot
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    zero_shot_pred = result["labels"][0]
    
    # Logistic regression
    test_X = vectorizer.transform([clause])
    lr_pred = model_lr.predict(test_X)[0]
    
    print("Zero-shot:", zero_shot_pred)
    print("Logistic Regression:", lr_pred)


CLAUSE:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary con
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are 
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 2. Customize and complete all areas noted with an "INSERT...", which require content to be inserted by University of Toronto faculty or staff, as appopriate.
Zero-shot: high risk
Logistic Regression: high risk

CLAUSE:
 3. Notify the authorized signatory if changes have been made to this template by the third party prior to execution because proposed amendments require legal review.
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 4. Once the NDA is 

# 4. Feature 3: Jargon Explanation

## Baseline
Simple prompt-based explanation.

## Improved
Enhanced prompt and decoding for better clarity.

## Comparison
Outputs are compared to evaluate improvement.

In [19]:
def explain_clause_v1(text):
    input_text = "explain in simple terms: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
print("ORIGINAL:\n", sample_clause)
print("\nBASELINE EXPLANATION:\n", explain_clause_v1(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

BASELINE EXPLANATION:
 : 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.


In [21]:
def explain_clause_v2(text):
    input_text = "rewrite this legal sentence in very simple plain English for a non-lawyer: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40,
        num_beams=5,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [22]:
print("\nIMPROVED EXPLANATION:\n", explain_clause_v2(sample_clause))


IMPROVED EXPLANATION:
 1. Consult your academic divisions guidelines regarding the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are typically not typically authorized to execute an NDA.
